<a href="https://colab.research.google.com/github/martatolos/eae-dsaa/blob/main/nlp_tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Calling, Agents, and RAG

> **Goal of the session:**
>
> - Understand how to give an LLM external capabilities through tool calling.
> - Build a simple agent that chains multiple tool calls autonomously.
> - Implement a lightweight Retrieval-Augmented Generation (RAG) pipeline using only tool calling and OpenAI embeddings — no external frameworks.
>
> **Scope of the session**
>
> - Section 1: What is tool calling and how does it work?
> - Section 2: The agent loop — letting the model drive multi-step reasoning.
> - Section 3: RAG via tool calling — grounding answers in your own documents.

## Setup

#### Dependencies

- `ipython`
- `openai` 2.40.0
- `numpy`
- `python-dotenv`

In [ ]:
%pip install ipython openai==2.40.0 numpy python-dotenv

### Imports

In [ ]:
import json
import os

import dotenv
import numpy as np
from IPython.display import Markdown, display
from openai import OpenAI

### API Key

Add your OpenAI API key in the cell below or create a `.env` file in the same directory as this notebook:

```
OPENAI_API_KEY=your_openai_api_key
```

> [!Warning]
> Make sure you do not save or commit the file without removing your API key.

In [ ]:
open_ai_key = None  # Add your OpenAI API key here
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", open_ai_key)

### Helper functions

In [ ]:
def render_markdown(text: str) -> None:
    display(Markdown(text))


def show_completion(prompt: str, model_name: str = "gpt-4.1-nano") -> None:
    response = OpenAI().responses.create(model=model_name, input=prompt)
    render_markdown(response.output[0].content[0].text)

---

## 1. What is Tool Calling?

### 1.1 The Concept

By default, an LLM can only produce text — it cannot query a database, look up a price, or perform a precise calculation on its own.

**Tool calling** solves this by letting you register functions the model can invoke. When the model decides it needs external information, it responds with a *tool call request* rather than a final answer. Your code then:

1. Detects the tool call in the response.
2. Executes the function with the arguments the model provided.
3. Feeds the result back to the model.
4. The model uses the result to compose its final answer.

```
User prompt ──► LLM decides to call a tool
                  │
                  ▼
             Your code runs the function
                  │
                  ▼
             Result sent back to LLM
                  │
                  ▼
             LLM produces final answer
```

Crucially, **the model does not execute any code itself** — it only tells you what it wants to call and with what arguments.

### 1.2 Defining a Tool

A tool is described using a JSON Schema: you specify its `name`, a plain-English `description` (the model reads this to decide when to use the tool), and the `parameters` it accepts.

Let's define a `get_stock_price` tool — a realistic example for a business context.

In [ ]:
get_stock_price_tool = {
    "type": "function",
    "name": "get_stock_price",
    "description": (
        "Returns the current stock price for a given ticker symbol. "
        "Use this tool whenever the user asks about a company's share price."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
            },
            "currency": {
                "type": "string",
                "description": "Currency for the price, e.g. USD, EUR. Defaults to USD.",
                "default": "USD"
            }
        },
        "required": ["ticker"]
    }
}

Now let's send a prompt that requires this information. Notice that the model **does not make up a price** — instead, it returns a tool call request.

In [ ]:
response = OpenAI().responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": "What is the current stock price of Apple in EUR?"}],
    tools=[get_stock_price_tool]
)

tool_call = response.output[0]
print(f"Response type : {tool_call.type}")
print(f"Tool requested: {tool_call.name}")
print(f"Arguments     : {tool_call.arguments}")

### 1.3 Executing the Tool and Returning the Result

We now:
1. Parse the arguments the model sent.
2. Run our actual Python function.
3. Feed the result back so the model can compose a final answer.

In [ ]:
# Mock implementation — in production this would call a real market data API
def get_stock_price(ticker: str, currency: str = "USD") -> dict:
    prices = {
        "AAPL": {"USD": 189.50, "EUR": 175.20},
        "MSFT": {"USD": 415.30, "EUR": 383.60},
        "TSLA": {"USD": 248.10, "EUR": 229.30},
        "NVDA": {"USD": 875.40, "EUR": 808.80},
        "AMZN": {"USD": 198.70, "EUR": 183.50},
    }
    price = prices.get(ticker.upper(), {}).get(currency.upper(), "N/A")
    return {"ticker": ticker.upper(), "price": price, "currency": currency.upper()}


args = json.loads(tool_call.arguments)
result = get_stock_price(**args)
print(f"Tool result: {result}")

In [ ]:
# Feed the tool result back — the model now has all it needs to answer
response2 = OpenAI().responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "user", "content": "What is the current stock price of Apple in EUR?"},
        tool_call,  # the model's function_call output item
        {"type": "function_call_output", "call_id": tool_call.call_id, "output": json.dumps(result)}
    ],
    tools=[get_stock_price_tool]
)

render_markdown(response2.output[0].content[0].text)

### 1.4 Exercise

Work through the three tasks below. Each task has its own code cell.

**Task 1 — Currency converter**

Define a `convert_currency(amount, from_currency, to_currency)` tool backed by a small dict of mock exchange rates. Ask "How much is 500 EUR in JPY?" and run the full tool-calling loop (define tool → first call → execute function → second call → final answer).

In [ ]:
# Write your tool definition, mock function, and tool-calling loop here

<details>
<summary>Example Solution</summary>

```python
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates = {"EUR": 1.0, "USD": 1.084, "JPY": 163.5, "GBP": 0.858, "CHF": 0.962}
    converted = round(amount / rates[from_currency.upper()] * rates[to_currency.upper()], 2)
    return {"amount": converted, "currency": to_currency.upper()}

currency_tool = {
    "type": "function", "name": "convert_currency",
    "description": "Convert an amount from one currency to another.",
    "parameters": {
        "type": "object",
        "properties": {
            "amount":        {"type": "number"},
            "from_currency": {"type": "string"},
            "to_currency":   {"type": "string"}
        },
        "required": ["amount", "from_currency", "to_currency"]
    }
}

client = OpenAI()
prompt_text = "How much is 500 EUR in JPY?"
r1 = client.responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": prompt_text}],
    tools=[currency_tool]
)
tc = r1.output[0]
result = convert_currency(**json.loads(tc.arguments))
r2 = client.responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "user", "content": prompt_text},
        tc,
        {"type": "function_call_output", "call_id": tc.call_id, "output": json.dumps(result)}
    ],
    tools=[currency_tool]
)
render_markdown(r2.output[0].content[0].text)
```

</details>

**Task 2 — Product catalogue lookup**

Define a `lookup_product(product_id)` tool backed by a Python dict with at least 4 products (each with a name, category, and price). Ask a question that requires looking up **two** products (e.g., "Compare the price of product P001 and product P002"). Observe that the model issues two separate tool calls.

In [ ]:
# Write your tool definition, mock function, and tool-calling loop here

<details>
<summary>Example Solution</summary>

```python
catalogue = {
    "P001": {"name": "Wireless Keyboard",       "price": 49.99},
    "P002": {"name": "USB-C Hub",               "price": 34.99},
    "P003": {"name": "Noise-Cancel Headphones", "price": 129.99},
    "P004": {"name": "Webcam HD",               "price": 79.99},
}

def lookup_product(product_id: str) -> dict:
    return catalogue.get(product_id.upper(), {"error": "Product not found"})

prod_tool = {
    "type": "function", "name": "lookup_product",
    "description": "Look up a product by its ID and return its name and price.",
    "parameters": {"type": "object",
                   "properties": {"product_id": {"type": "string"}},
                   "required": ["product_id"]}
}

answer = run_agent(
    "Compare the price of product P001 and product P002.",
    tools=[prod_tool],
    tool_handlers={"lookup_product": lookup_product}
)
render_markdown(answer)
```

</details>

**Task 3 — Date calculator**

Define a `days_between(date1, date2)` tool that uses Python's `datetime` module to compute the number of days between two ISO-format dates. Ask "How many days are there between 1 January 2024 and 15 March 2025?" and complete the tool-calling loop.

In [ ]:
# Write your tool definition, function, and tool-calling loop here

<details>
<summary>Example Solution</summary>

```python
from datetime import date as dt_date

def days_between(date1: str, date2: str) -> dict:
    d1 = dt_date.fromisoformat(date1)
    d2 = dt_date.fromisoformat(date2)
    return {"days": abs((d2 - d1).days)}

date_tool = {
    "type": "function", "name": "days_between",
    "description": "Calculate the number of days between two ISO-format dates (YYYY-MM-DD).",
    "parameters": {
        "type": "object",
        "properties": {
            "date1": {"type": "string", "description": "Start date in YYYY-MM-DD format."},
            "date2": {"type": "string", "description": "End date in YYYY-MM-DD format."}
        },
        "required": ["date1", "date2"]
    }
}

client = OpenAI()
prompt_text = "How many days are there between 1 January 2024 and 15 March 2025?"
r1 = client.responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": prompt_text}],
    tools=[date_tool]
)
tc = r1.output[0]
result = days_between(**json.loads(tc.arguments))
r2 = client.responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "user", "content": prompt_text},
        tc,
        {"type": "function_call_output", "call_id": tc.call_id, "output": json.dumps(result)}
    ],
    tools=[date_tool]
)
render_markdown(r2.output[0].content[0].text)
```

</details>

---

## 2. The Agent Loop

### 2.1 What Makes Something an "Agent"?

In the previous section, we handled a single tool call manually. A real **agent** automates this: it runs a loop where the model keeps calling tools until it has gathered everything it needs to answer.

```
User prompt
    │
    ▼
LLM response ──► tool call? ──yes──► execute tool ──► feed result back ──┐
    │                                                                      │
    no                                                                     │
    │◄─────────────────────────────────────────────────────────────────────┘
    ▼
Final answer
```

No external framework is needed — the loop is just a `while` block in Python.

### 2.2 Building the Agent Loop

In [ ]:
def run_agent(prompt: str, tools: list, tool_handlers: dict, model: str = "gpt-4.1-nano") -> str:
    """Run an agent that calls tools until it reaches a final text answer.

    :param prompt: The user's question.
    :param tools: List of tool schema dicts.
    :param tool_handlers: Dict mapping tool name → Python callable.
    :param model: OpenAI model to use.
    :return: Final text answer from the model.
    """
    client = OpenAI()
    input_items = [{"role": "user", "content": prompt}]

    while True:
        response = client.responses.create(model=model, input=input_items, tools=tools)

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tool calls — model has produced its final answer
            return response.output[0].content[0].text

        # Execute every tool call the model requested and append results
        for tc in tool_calls:
            input_items.append(tc)  # record model's request in history
            args = json.loads(tc.arguments)
            result = tool_handlers[tc.name](**args)
            print(f"  [tool] {tc.name}({args}) → {result}")
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": json.dumps(result)
            })

Let's demonstrate with a multi-step question that requires **two product lookups followed by a currency conversion** — three tool calls in total.

In [ ]:
# Reuse the tools from the exercise above — define them here if needed
def lookup_product(product_id: str) -> dict:
    catalogue = {
        "P001": {"name": "Wireless Keyboard", "price_usd": 49.99},
        "P002": {"name": "USB-C Hub",          "price_usd": 34.99},
        "P003": {"name": "Noise-Cancel Headphones", "price_usd": 129.99},
    }
    return catalogue.get(product_id.upper(), {"error": "Product not found"})

def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates = {"USD": 1.0, "EUR": 0.924, "GBP": 0.792, "JPY": 149.50}
    converted = round(amount * rates.get(to_currency.upper(), 1) / rates.get(from_currency.upper(), 1), 2)
    return {"amount": converted, "currency": to_currency.upper()}

tools = [
    {
        "type": "function", "name": "lookup_product",
        "description": "Look up a product by its ID and return its name and price in USD.",
        "parameters": {"type": "object", "properties": {"product_id": {"type": "string"}}, "required": ["product_id"]}
    },
    {
        "type": "function", "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount":        {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency":   {"type": "string"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
]

handlers = {"lookup_product": lookup_product, "convert_currency": convert_currency}

answer = run_agent(
    prompt="What is the combined price of products P001 and P002 converted to EUR?",
    tools=tools,
    tool_handlers=handlers
)
render_markdown(answer)

### 2.3 Controlling Tool Use

By default the model decides when to use a tool (`tool_choice="auto"`). You can override this:

| Setting | Effect |
|---|---|
| `"auto"` | Model decides whether to call a tool (default) |
| `"required"` | Model must call at least one tool |
| `{"type": "function", "name": "..."}` | Force a specific tool |

In [ ]:
client = OpenAI()
prompt = "What is the current stock price of Tesla?"

# auto — model calls the tool because it needs the data
r_auto = client.responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": prompt}],
    tools=[get_stock_price_tool],
    tool_choice="auto"
)
print("auto     →", r_auto.output[0].type)

# required — model must call a tool even if it thinks it knows the answer
r_req = client.responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": "What is 2 + 2?"}],
    tools=[get_stock_price_tool],
    tool_choice="required"
)
print("required →", r_req.output[0].type, "|", r_req.output[0].name)

# specific tool forced by name
r_spec = client.responses.create(
    model="gpt-4.1-nano",
    input=[{"role": "user", "content": "Tell me something interesting about NVIDIA."}],
    tools=[get_stock_price_tool],
    tool_choice={"type": "function", "name": "get_stock_price"}
)
print("specific →", r_spec.output[0].type, "|", r_spec.output[0].arguments)

### 2.4 Exercise

**Task 1 — Research assistant**

Define two tools:
- `search_articles(topic: str)` → returns a hardcoded list of 3 short article summaries on that topic.
- `get_article_details(article_id: str)` → returns the full text of one article (a few sentences).

Ask: *"Give me a brief overview of recent AI regulation in Europe."* Run it through `run_agent()` and observe how the model chains both tools.

In [ ]:
# Write your tools, handlers, and run_agent() call here

<details>
<summary>Example Solution</summary>

```python
articles_db = {
    "ai regulation": [
        {"id": "A1", "summary": "EU AI Act: risk-based regulation banning certain AI uses."},
        {"id": "A2", "summary": "GDPR and AI: new guidelines on training data and automated decisions."},
        {"id": "A3", "summary": "AI liability directive: easier compensation for AI-caused harm."},
    ]
}

def search_articles(topic: str) -> list:
    for key in articles_db:
        if key in topic.lower():
            return articles_db[key]
    return [{"id": "none", "summary": "No articles found."}]

def get_article_details(article_id: str) -> dict:
    for articles in articles_db.values():
        for a in articles:
            if a["id"] == article_id:
                return a
    return {"error": "Article not found"}

tools = [
    {"type": "function", "name": "search_articles",
     "description": "Search for recent news articles on a given topic.",
     "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]}},
    {"type": "function", "name": "get_article_details",
     "description": "Get the full summary of a specific article by its ID.",
     "parameters": {"type": "object", "properties": {"article_id": {"type": "string"}}, "required": ["article_id"]}}
]

answer = run_agent(
    "Give me a brief overview of recent AI regulation in Europe.",
    tools=tools,
    tool_handlers={"search_articles": search_articles, "get_article_details": get_article_details}
)
render_markdown(answer)
```

</details>

**Task 2 — Trip planner**

Define two tools:
- `get_flight_price(origin: str, destination: str)` → returns a mock round-trip price in EUR.
- `list_attractions(city: str)` → returns a list of 3–4 top attractions for a city.

Ask: *"I'm flying from Madrid to Lisbon — what will it cost and what are the top things to do there?"* and run the agent loop.

In [ ]:
# Write your tools, handlers, and run_agent() call here

<details>
<summary>Example Solution</summary>

```python
def get_flight_price(origin: str, destination: str) -> dict:
    prices = {("madrid", "lisbon"): 89, ("barcelona", "paris"): 120}
    key = (origin.lower(), destination.lower())
    price = prices.get(key, prices.get((destination.lower(), origin.lower()), 149))
    return {"origin": origin, "destination": destination, "price_eur": price, "type": "round-trip"}

def list_attractions(city: str) -> list:
    db = {
        "lisbon": ["Belem Tower", "Alfama district", "Jeronimos Monastery", "LX Factory"],
        "madrid": ["Prado Museum", "Retiro Park", "Puerta del Sol"],
    }
    return db.get(city.lower(), ["City centre", "Historic old town", "Local market"])

tools = [
    {"type": "function", "name": "get_flight_price",
     "description": "Get the round-trip flight price in EUR between two cities.",
     "parameters": {"type": "object",
                    "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}},
                    "required": ["origin", "destination"]}},
    {"type": "function", "name": "list_attractions",
     "description": "List the top tourist attractions in a city.",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}
]

answer = run_agent(
    "I'm flying from Madrid to Lisbon — what will it cost and what are the top things to do there?",
    tools=tools,
    tool_handlers={"get_flight_price": get_flight_price, "list_attractions": list_attractions}
)
render_markdown(answer)
```

</details>

**Task 3 — Budget calculator**

Define two tools:
- `get_product_price(name: str)` → returns a unit price from a small hardcoded catalogue.
- `calculate_total(items: list, quantities: list)` → returns the sum of `price × quantity` for each item.

Ask: *"What would it cost to buy 3 units of the wireless keyboard and 2 USB-C hubs?"* and run the agent loop.

In [ ]:
# Write your tools, handlers, and run_agent() call here

<details>
<summary>Example Solution</summary>

```python
def get_product_price(name: str) -> dict:
    prices = {"wireless keyboard": 49.99, "usb-c hub": 34.99,
              "noise-cancel headphones": 129.99, "webcam hd": 79.99}
    price = prices.get(name.lower())
    if price is None:
        return {"error": f"Product '{name}' not found"}
    return {"name": name, "unit_price": price, "currency": "USD"}

def calculate_total(items: list, quantities: list) -> dict:
    total = sum(item["unit_price"] * qty for item, qty in zip(items, quantities))
    return {"total": round(total, 2), "currency": "USD"}

tools = [
    {"type": "function", "name": "get_product_price",
     "description": "Look up the unit price of a product by its name.",
     "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}},
    {"type": "function", "name": "calculate_total",
     "description": "Calculate the total cost given a list of items (each with unit_price) and quantities.",
     "parameters": {
         "type": "object",
         "properties": {
             "items":      {"type": "array", "items": {"type": "object"}},
             "quantities": {"type": "array", "items": {"type": "integer"}}
         },
         "required": ["items", "quantities"]
     }}
]

answer = run_agent(
    "What would it cost to buy 3 wireless keyboards and 2 USB-C hubs?",
    tools=tools,
    tool_handlers={"get_product_price": get_product_price, "calculate_total": calculate_total}
)
render_markdown(answer)
```

</details>

---

## 3. RAG via Tool Calling

### 3.1 Why RAG?

LLMs have a knowledge cut-off and no access to private or proprietary documents. **Retrieval-Augmented Generation (RAG)** solves this:

1. Store your documents as text chunks with corresponding vector embeddings.
2. When the user asks a question, find the most relevant chunks (by semantic similarity).
3. Inject those chunks into the prompt so the model can answer from your content.

Traditionally this requires a dedicated vector database and an orchestration framework. In this section we build the same pipeline using **only the OpenAI SDK and NumPy** — exposing retrieval as a tool the model calls when it needs context.

### 3.2 Building a Lightweight Vector Store

In [ ]:
# Knowledge base: FAQ for a fictional SaaS company "NovaTech Analytics"
knowledge_base = [
    "NovaTech Analytics offers three pricing tiers: Starter (€29/month, up to 5 users), "
    "Professional (€99/month, up to 25 users), and Enterprise (custom pricing, unlimited users).",

    "The Professional plan includes advanced dashboards, API access, and priority email support. "
    "The Starter plan is limited to basic reporting and community support only.",

    "NovaTech provides a 14-day free trial for both Starter and Professional plans. "
    "No credit card is required to start the trial.",

    "Data is stored in EU-based data centres and is encrypted at rest using AES-256. "
    "NovaTech is fully GDPR-compliant and undergoes annual ISO 27001 audits.",

    "Customers can cancel their subscription at any time. "
    "Refunds are issued on a pro-rata basis for unused days in the current billing period.",

    "NovaTech integrates natively with Salesforce, HubSpot, Slack, and Google Workspace. "
    "A REST API and webhook support are available on Professional and Enterprise plans.",

    "Technical support is available Monday to Friday, 09:00–18:00 CET. "
    "Enterprise customers receive 24/7 dedicated support with a guaranteed 4-hour response SLA.",
]

print(f"Knowledge base loaded: {len(knowledge_base)} chunks")

In [ ]:
# Embed all chunks once at startup — store as list of (chunk_text, embedding_vector) pairs
client = OpenAI()

embed_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=knowledge_base
)

chunk_embeddings = [
    (chunk, item.embedding)
    for chunk, item in zip(knowledge_base, embed_response.data)
]

print(f"Embedded {len(chunk_embeddings)} chunks (vector length: {len(chunk_embeddings[0][1])})")

In [ ]:
def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search_knowledge_base(query: str, top_k: int = 3) -> list[str]:
    """Embed the query and return the top-k most similar chunks."""
    query_vec = OpenAI().embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    scored = [
        (cosine_similarity(query_vec, emb), chunk)
        for chunk, emb in chunk_embeddings
    ]
    scored.sort(reverse=True)
    return [chunk for _, chunk in scored[:top_k]]


# Quick sanity check
results = search_knowledge_base("How much does the Professional plan cost?")
for i, chunk in enumerate(results, 1):
    print(f"[{i}] {chunk[:100]}...")

### 3.3 Exposing Retrieval as a Tool

In [ ]:
search_tool = {
    "type": "function",
    "name": "search_knowledge_base",
    "description": (
        "Search the NovaTech Analytics knowledge base for information about pricing, features, "
        "integrations, security, or support. Use this tool before answering any question about NovaTech — "
        "do not rely on your own knowledge."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A natural-language question or keyword to search for."
            },
            "top_k": {
                "type": "integer",
                "description": "Number of chunks to retrieve. Defaults to 3.",
                "default": 3
            }
        },
        "required": ["query"]
    }
}

In [ ]:
rag_handlers = {"search_knowledge_base": search_knowledge_base}

questions = [
    "Does NovaTech offer a free trial? Do I need a credit card?",
    "Which plan should I choose if I need API access and have a team of 15 people?",
    "What happens to my data if I cancel my subscription?",
]

for question in questions:
    render_markdown(f"**Q: {question}**")
    answer = run_agent(
        prompt=question,
        tools=[search_tool],
        tool_handlers=rag_handlers
    )
    render_markdown(answer)
    render_markdown("---")

> **Key insight:** The model calls `search_knowledge_base` before answering — it does not guess from its training data. This means answers stay accurate even as the knowledge base changes, without retraining the model.

### 3.4 Exercise

**Task 1 — Custom knowledge base**

Replace the NovaTech FAQ with 5–6 short paragraphs about a topic of your choice (e.g., your company's HR policies, a product manual, a course syllabus). Rebuild the embeddings and run at least 3 questions that span different paragraphs. Verify that the retrieved chunks are relevant.

In [ ]:
# Define your own knowledge_base list, re-run the embedding cell, then test your questions here
# Write your code here

<details>
<summary>Example Solution</summary>

```python
my_knowledge_base = [
    "Employees are entitled to 25 days of annual leave per year, accruing at 2.08 days per month.",
    "Remote work is permitted up to 3 days per week. Core hours (10:00-16:00 CET) must be observed.",
    "All expenses above 50 EUR require a receipt and manager approval prior to reimbursement.",
    "Performance reviews run twice a year: June and December. Ratings range from 1 to 5.",
    "New employees have a 3-month probationary period with one week notice for either party.",
    "Full-time employees receive health insurance from day one. Part-timers over 20 hours/week also qualify.",
]

embed_resp = OpenAI().embeddings.create(model="text-embedding-3-small", input=my_knowledge_base)
my_embeddings = [(chunk, item.embedding) for chunk, item in zip(my_knowledge_base, embed_resp.data)]

def search_hr_kb(query: str, top_k: int = 3) -> list:
    q_vec = OpenAI().embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding
    scored = [(cosine_similarity(q_vec, emb), chunk) for chunk, emb in my_embeddings]
    scored.sort(reverse=True)
    return [chunk for _, chunk in scored[:top_k]]

hr_tool = {
    "type": "function", "name": "search_hr_kb",
    "description": "Search the HR policy knowledge base. Use before answering any HR question.",
    "parameters": {"type": "object",
                   "properties": {"query": {"type": "string"}, "top_k": {"type": "integer", "default": 3}},
                   "required": ["query"]}
}

for q in ["How much annual leave do I get?", "Can I work from home every day?", "When is my next performance review?"]:
    render_markdown(f"**Q: {q}**")
    ans = run_agent(q, tools=[hr_tool], tool_handlers={"search_hr_kb": search_hr_kb})
    render_markdown(ans + "\n---")
```

</details>

**Task 2 — Grounded answers with citations**

Modify `search_knowledge_base` to return each chunk together with its index number (e.g., `[0] chunk text`). Update the agent prompt with an instruction such as *"Always cite the chunk index(es) you used in your answer"*. Run the same questions and observe whether the model correctly attributes its answers.

In [ ]:
# Write your modified search function and agent call here

<details>
<summary>Example Solution</summary>

```python
def search_kb_cited(query: str, top_k: int = 3) -> list:
    q_vec = OpenAI().embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding
    scored = [(cosine_similarity(q_vec, emb), i, chunk) for i, (chunk, emb) in enumerate(chunk_embeddings)]
    scored.sort(reverse=True)
    return [{"chunk_index": i, "text": chunk} for _, i, chunk in scored[:top_k]]

cited_tool = {
    "type": "function", "name": "search_kb_cited",
    "description": (
        "Search the NovaTech knowledge base. Returns chunks with their index. "
        "Always cite the chunk_index(es) you used at the end of your answer."
    ),
    "parameters": {"type": "object",
                   "properties": {"query": {"type": "string"}, "top_k": {"type": "integer", "default": 3}},
                   "required": ["query"]}
}

for q in ["Does NovaTech offer a free trial?", "What security certifications does NovaTech hold?"]:
    render_markdown(f"**Q: {q}**")
    ans = run_agent(q, tools=[cited_tool], tool_handlers={"search_kb_cited": search_kb_cited})
    render_markdown(ans + "\n---")
```

</details>